# Proyecto Integrador 2 · Parte IV — Análisis de enlaces: PageRank y HITS
## Quién importa en la red de compras: PageRank sobre un grafo dirigido y HITS sobre el grafo entidad–proveedor

**Qué hace este notebook**
1. Construye en **Spark** un grafo dirigido **«proveedor A pierde frente a proveedor B»** y lo justifica.
2. Calcula **PageRank** con teletransporte (la *Google matrix* de clase) con iteración de potencia (*power iteration*) en Spark, e interpreta los de mayor rango.
3. Mide la **sensibilidad al factor de amortiguamiento β** (cuánto cambia el ranking si cambia β).
4. Calcula **HITS** sobre el grafo entidad→proveedor: las **entidades** son *hubs* y los **proveedores** son *authorities*.
5. Guarda archivos ligeros para la página web.

**Entrada:** Parquet del Proyecto 1 y los Parquet del notebook 1 (`proveedor_dim`, `entidad_dim`, `interacciones_train`, `features_proveedor`).
**Salida:** `grafo_derrotas` (Parquet) y archivos `pr_*`, `hits_*`, `p4_resumen.json` para la página web.

**Pregunta de interés que responde (P3):** ¿qué actores concentran el mercado? Aquí se responde con **PageRank y HITS**; las comunidades se ven en el notebook 8.

**Importante:** el resultado es una **señal descriptiva**. Un proveedor con PageRank alto es uno que gana frente a rivales que también ganan mucho; no es una acusación ni una prueba de irregularidades.

**Qué enseñó la clase y qué es extra**
- PageRank, Google matrix, iteración de potencia, *dead ends*: **visto en clase**.
- HITS: la clase solo lo menciona en la lista de extensiones. Se calcula con la misma técnica de iteración de potencia; es el único método extra de este notebook.

## 0. Configuración

In [1]:
import os, json, time
import numpy as np
import pandas as pd

USAR_DRIVE = False   # True si corres en Colab con Drive
RUTA_PARQUET = os.environ.get("P2_PARQUET", "/content/drive/MyDrive/proyecto_integrador_seace/parquet")
RUTA_P2      = os.environ.get("P2_DATOS",   "/content/drive/MyDrive/proyecto_integrador_seace/p2")
WEB          = os.environ.get("P2_WEB",     "/content/drive/MyDrive/proyecto_integrador_seace/web_artefactos/p2")

MIN_PART = 5          # mismo criterio de nodo que el notebook 2: al menos 5 participaciones
TOPE     = 30         # mismo tope de postores que el notebook 2
BETA     = 0.85       # factor de amortiguamiento principal (el valor usado en clase)
TOL      = 1e-9       # se detiene cuando el cambio total entre iteraciones es menor que esto
MAX_ITER = 400    # tope de iteraciones (con β = 0.95 hacen falta unas 300)
BETAS    = [0.50, 0.70, 0.80, 0.85, 0.90, 0.95]   # valores para el análisis de sensibilidad

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(WEB, exist_ok=True)

def guardar_json(nombre, obj):
    with open(f"{WEB}/{nombre}", "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print("guardado:", nombre)

def guardar_csv(nombre, df):
    df.to_csv(f"{WEB}/{nombre}", index=False)
    print("guardado:", nombre, len(df), "filas")

## 1. Spark y lectura

In [2]:
# Solo Colab: !pip install -q pyspark==3.5.3
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]").appName("p2_pagerank")
         .config("spark.driver.memory", "4g").config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

dim   = spark.read.parquet(f"{RUTA_P2}/proveedor_dim").select("proveedor_idx", "ruc")
post  = spark.read.parquet(f"{RUTA_PARQUET}/postores").select("release_id", "ruc")
adj   = spark.read.parquet(f"{RUTA_PARQUET}/adjudicaciones").select("release_id", F.col("proveedor_ruc").alias("ruc"))
train = spark.read.parquet(f"{RUTA_P2}/interacciones_train").select("entidad_idx", "proveedor_idx", "n_adjudicaciones")

# un proveedor por proceso (sin repetidos). Al unir con proveedor_dim se van los consorcios (ids de 7 dígitos)
participa = post.join(dim, "ruc").select("release_id", "proveedor_idx").distinct().cache()
gano      = adj.join(dim, "ruc").select("release_id", "proveedor_idx").distinct().withColumn("gano", F.lit(1))
print("participaciones (proceso, proveedor):", participa.count())

To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


participaciones (proceso, proveedor): 2371613


## 2. El grafo dirigido «pierde frente a»

**Por qué un grafo dirigido.** PageRank necesita enlaces con dirección: una página «vota» por otra. En compras públicas hay un enlace natural con dirección: **si A y B ofertaron en el mismo proceso y ganó B, A «apunta» a B.** Cada derrota es un voto: *«B fue mejor que yo»*.

- **Nodo:** un proveedor con al menos `MIN_PART` participaciones (los mismos del grafo del notebook 2).
- **Arista A → B:** A ofertó, **no ganó**, y B **sí ganó** en el mismo proceso. El **peso** es en cuántos procesos pasó eso.
- **Quién gana:** quien figura en las adjudicaciones del proceso. Si el proceso tiene varios lotes y A ganó alguno, A cuenta como ganador y no genera aristas de derrota.
- **Se usa el mismo tope de postores** que el notebook 2 (`TOPE = 30`): un proceso con muchísimos postores no dice casi nada de quién «compite» con quién.

**Qué significa PageRank aquí:** un proveedor tiene rango alto si **gana frente a muchos rivales y esos rivales a su vez ganan** (se parece a un ranking de torneo). Un proveedor que **nunca pierde** no tiene aristas de salida: es un *dead end* (callejón sin salida) del modelo de clase, y se maneja con la teletransportación.

**Limitaciones:** solo se ve lo que está en SEACE con RUC real; se dejan fuera consorcios y adjudicaciones sin RUC (notebook 1). Ganar un proceso no es lo mismo que ser «mejor»: el precio, los requisitos y el momento influyen.

In [3]:
nodos_ok = (participa.groupBy("proveedor_idx").agg(F.count("*").alias("n_part"))
            .filter(F.col("n_part") >= MIN_PART).select("proveedor_idx"))
procesos_ok = (participa.groupBy("release_id").agg(F.count("*").alias("k"))
               .filter(F.col("k") <= TOPE).select("release_id"))

pf = (participa.join(procesos_ok, "release_id").join(F.broadcast(nodos_ok), "proveedor_idx")
      .join(gano, ["release_id", "proveedor_idx"], "left")
      .withColumn("gano", F.coalesce("gano", F.lit(0))))

ganadores  = pf.filter("gano = 1").select("release_id", F.col("proveedor_idx").alias("ganador"))
perdedores = pf.filter("gano = 0").select("release_id", F.col("proveedor_idx").alias("perdedor"))

arcos = (perdedores.join(ganadores, "release_id")
         .groupBy("perdedor", "ganador").agg(F.count("*").alias("peso")))
arcos.write.mode("overwrite").parquet(f"{RUTA_P2}/grafo_derrotas")
arcos = spark.read.parquet(f"{RUTA_P2}/grafo_derrotas").cache()

nodos = nodos_ok.select(F.col("proveedor_idx").alias("nodo")).cache()
N = nodos.count()
n_arcos = arcos.count()
con_salida  = arcos.select(F.col("perdedor").alias("nodo")).distinct()
con_entrada = arcos.select(F.col("ganador").alias("nodo")).distinct()
n_sin_salida  = nodos.join(con_salida, "nodo", "left_anti").count()    # nunca pierden: dead ends
n_sin_entrada = nodos.join(con_entrada, "nodo", "left_anti").count()   # nadie les pierde
print(f"nodos: {N:,} | aristas dirigidas: {n_arcos:,}")
print(f"dead ends (nunca pierden, sin aristas de salida): {n_sin_salida:,} ({100*n_sin_salida/N:.1f}%)")
print(f"sin aristas de entrada (nadie pierde frente a ellos): {n_sin_entrada:,} ({100*n_sin_entrada/N:.1f}%)")
print("arcos con peso ≥ 2:", arcos.filter("peso >= 2").count())

nodos: 43,115 | aristas dirigidas: 508,252
dead ends (nunca pierden, sin aristas de salida): 5,110 (11.9%)
sin aristas de entrada (nadie pierde frente a ellos): 24,754 (57.4%)


arcos con peso ≥ 2: 117205


### Lectura
- Hay **muchos proveedores sin aristas de salida** (nunca pierden en los procesos que cuentan): suelen ser proveedores con pocas participaciones y todas ganadas. Es justo el caso de *dead end* que la clase resuelve con la teletransportación.
- La mayoría de los proveedores **no recibe** aristas de entrada: nadie pierde frente a ellos. El voto se concentra en una minoría que gana con frecuencia.

## 3. PageRank en Spark

**Fórmula de clase (Google matrix):** `r = β·M·r + (1 − β)/N`

- `r` es el PageRank de cada nodo (suma 1).
- `M` es la matriz de transición: de A salen sus votos repartidos **en proporción al peso** de cada arista.
- `β` es el **factor de amortiguamiento** (*damping*): con probabilidad β el «caminante» sigue una arista y con probabilidad `1 − β` se teletransporta a un nodo al azar.
- **Dead ends:** un nodo sin aristas de salida reparte su rango **entre todos los nodos por igual** (equivale a teletransportarse siempre).

**Iteración de potencia:** se parte de `r = 1/N` y se repite la ecuación hasta que `r` casi no cambia. El **cambio** de una iteración es `Σ |r_nuevo − r|`.

En Spark cada iteración es: *unir aristas con `r` → multiplicar por el peso normalizado → sumar por destino*.

In [4]:
sal = arcos.groupBy("perdedor").agg(F.sum("peso").alias("sal"))
M = (arcos.join(sal, "perdedor")
          .select(F.col("perdedor").alias("src"), F.col("ganador").alias("dst"), (F.col("peso") / F.col("sal")).alias("m"))
          .cache())
dead = nodos.join(con_salida, "nodo", "left_anti").cache()   # nodos sin aristas de salida
M.count(); dead.count()

def pagerank_spark(beta, tol=TOL, max_iter=MAX_ITER):
    """Devuelve (DataFrame pandas con proveedor_idx y pr, lista de cambios por iteración)."""
    r = nodos.withColumn("r", F.lit(1.0 / N)).localCheckpoint()
    cambios = []
    for it in range(1, max_iter + 1):
        masa_dead = r.join(dead, "nodo").agg(F.sum("r")).first()[0] or 0.0      # rango que reparten los dead ends
        flujo = (M.join(r, M.src == r.nodo).groupBy("dst").agg(F.sum(F.col("m") * F.col("r")).alias("f")))
        nuevo = (nodos.join(flujo, nodos.nodo == flujo.dst, "left")
                      .select("nodo", (beta * (F.coalesce("f", F.lit(0.0)) + masa_dead / N) + (1 - beta) / N).alias("r"))
                      .localCheckpoint())
        cambio = (r.alias("a").join(nuevo.alias("b"), "nodo").agg(F.sum(F.abs(F.col("a.r") - F.col("b.r")))).first()[0])
        cambios.append(cambio)
        r = nuevo
        if cambio < tol:
            break
    out = r.toPandas().rename(columns={"nodo": "proveedor_idx", "r": "pr"})
    return out, cambios

t0 = time.time()
pr, cambios = pagerank_spark(BETA)
print(f"β = {BETA}: {len(cambios)} iteraciones, {time.time()-t0:.0f} s | suma de r = {pr.pr.sum():.6f}")
pd.DataFrame({"iteracion": range(1, 9), "cambio": [f"{c:.2e}" for c in cambios[:8]]})

β = 0.85: 95 iteraciones, 73 s | suma de r = 1.000000


,iteracion,cambio
0,1,9.95e-01
1,2,2.66e-01
2,3,1.02e-01
3,4,5.12e-02
4,5,3.08e-02
5,6,2.06e-02
6,7,1.45e-02
7,8,1.05e-02


**Cómo leer la tabla:** el cambio cae casi en forma de línea recta en escala logarítmica: cada iteración reduce el error por un factor cercano a β. Por eso se llega a un cambio de `1e-9` en unas pocas decenas de iteraciones. La suma de `r` es 1 (nadie «pierde» rango).

### Verificación contra NetworkX
Para estar seguros de que el cálculo en Spark está bien, se compara con `networkx.pagerank` sobre el mismo grafo (NetworkX es solo una verificación; no es parte de la solución).

In [5]:
import networkx as nx
ar = arcos.toPandas()
Gd = nx.DiGraph()
Gd.add_nodes_from(nodos.toPandas().nodo.tolist())
Gd.add_weighted_edges_from(ar[["perdedor", "ganador", "peso"]].itertuples(index=False, name=None))
pr_nx = nx.pagerank(Gd, alpha=BETA, weight="weight", tol=1e-12, max_iter=500)
comparar = pr.set_index("proveedor_idx").pr
dif_max = max(abs(comparar[n] - v) for n, v in pr_nx.items())
print(f"diferencia máxima entre Spark y NetworkX: {dif_max:.2e}")
assert dif_max < 1e-8, "Spark y NetworkX no coinciden"

diferencia máxima entre Spark y NetworkX: 2.47e-09


## 4. Interpretar los de mayor rango
Se une el PageRank con datos del notebook 1: cuántas veces participó, cuántas veces ganó (`n_adjudicaciones`) y su tasa de éxito. Así se ve **qué agrega PageRank frente a simplemente contar victorias**.

In [6]:
feat = spark.read.parquet(f"{RUTA_P2}/features_proveedor").select(
    "proveedor_idx", "ruc", "nombre", "n_participaciones", "n_adjudicaciones", "tasa_exito", "n_entidades", "monto_total", "categoria_principal").toPandas()
t = pr.merge(feat, on="proveedor_idx", how="left")
t["rango_pr"] = t.pr.rank(ascending=False, method="first").astype(int)
t["rango_victorias"] = t.n_adjudicaciones.rank(ascending=False, method="first").astype(int)
t = t.sort_values("rango_pr").reset_index(drop=True)
t["pr_veces_uniforme"] = (t.pr * N).round(1)    # 1 = tanto como un nodo «promedio»
t["rivales_vencidos"] = t.proveedor_idx.map(ar.groupby("ganador").size()).fillna(0).astype(int)   # proveedores distintos que perdieron frente a él

print("Correlación de rangos (Spearman) entre PageRank y:")
for c in ["n_adjudicaciones", "n_participaciones", "n_entidades", "tasa_exito"]:
    print(f"  {c:20s} {t.pr.corr(t[c], method='spearman'):.3f}")
masa_top = t.pr.head(int(0.01 * N)).sum()
print(f"\nel 1% con más PageRank ({int(0.01*N)} proveedores) reúne {100*masa_top:.1f}% del PageRank total")
t.head(15)[["rango_pr", "nombre", "ruc", "pr_veces_uniforme", "n_participaciones", "n_adjudicaciones", "tasa_exito", "rango_victorias"]]

Correlación de rangos (Spearman) entre PageRank y:


  n_adjudicaciones     0.881
  n_participaciones    0.352
  n_entidades          0.873
  tasa_exito           0.797

el 1% con más PageRank (431 proveedores) reúne 16.7% del PageRank total


,rango_pr,nombre,ruc,pr_veces_uniforme,n_participaciones,n_adjudicaciones,tasa_exito,rango_victorias
0,1,GRUPO SANTA FE SOCIEDAD ANONIMA CERRADA - GRUP...,20511037001,312.8,3321,1427,0.429389,1
1,2,COMERCIAL FERRETERIA FATIMA SOCIEDAD ANONIMA C...,20449458436,75.5,960,218,0.227083,8
2,3,TMI-TYC SOCIEDAD ANONIMA CERRADA - TMI-TYC S.A.C.,20544585097,71.9,2355,255,0.108280,5
3,4,AMERICA MOVIL PERU S.A.C.,20467534026,64.8,1226,268,0.218597,4
4,5,FORZAMAT S.A.C.,20559644243,59.6,1725,158,0.091594,18
5,6,PROMAINGSA S.A.C.,20600616235,57.2,823,131,0.159174,27
6,7,MILNER INVERSIONES Y SERVICIOS EMPRESA INDIVID...,20600111150,56.0,1819,147,0.080814,22
7,8,IPESA S.A.C.,20101639275,54.1,542,140,0.258303,23
8,9,TRUYENQUE MALPARTIDA CESAR,10444256279,48.0,384,176,0.458333,12
9,10,DAILY TECHNOLOGY S.A.C.,20553404631,46.4,452,73,0.161504,91


In [7]:
# ¿En qué se diferencia PageRank de contar victorias? Proveedores con mucho más PageRank que lo que sugieren sus victorias, y al revés
t["dif_rangos"] = t.rango_victorias - t.rango_pr          # positivo: PageRank lo coloca mejor que el conteo de victorias
con_victorias = t[t.n_adjudicaciones >= 5]       # se comparan solo proveedores con al menos 5 victorias
top_n = con_victorias[con_victorias.rango_pr <= 2000]
print("Mejor ubicados por PageRank que por victorias (≥ 5 victorias, entre los 2,000 primeros por PageRank):")
display(top_n.sort_values("dif_rangos", ascending=False).head(8)[["rango_pr", "rango_victorias", "nombre", "n_participaciones", "n_adjudicaciones", "tasa_exito", "rivales_vencidos"]])
print("Peor ubicados por PageRank que por victorias (entre los 2,000 primeros por victorias):")
top_v = con_victorias[con_victorias.rango_victorias <= 2000]
display(top_v.sort_values("dif_rangos").head(8)[["rango_pr", "rango_victorias", "nombre", "n_participaciones", "n_adjudicaciones", "tasa_exito", "rivales_vencidos"]])

Mejor ubicados por PageRank que por victorias (≥ 5 victorias, entre los 2,000 primeros por PageRank):


,rango_pr,rango_victorias,nombre,n_participaciones,n_adjudicaciones,tasa_exito,rivales_vencidos
112,113,6091,CONSTRUCTORA Y CONSULTORA Y&A S.A.C.,24,5,0.208333,24
415,416,6035,KAMETSA CONSULTORES Y CONSTRUCTORES GENERALES ...,70,5,0.071429,79
602,603,6042,NAMEKU E.I.R.L,83,5,0.060241,33
627,628,6053,MM SERMACON S.A.C.,38,5,0.131579,52
419,420,5693,H&Y CRUZ SOCIEDAD ANONIMA CERRADA - H&Y CRUZ S...,41,5,0.121951,61
894,895,6168,EVAT INGENIERIA Y CONSTRUCCION S.A.C.,46,5,0.108696,50
784,785,6018,GRUPO SAGITARIO J & E SOCIEDAD ANONIMA CERRADA,29,5,0.172414,43
571,572,5767,A & S CONSTRUCCIÓN Y CONSULTORÍA E.I.R.L.,44,5,0.113636,50


Peor ubicados por PageRank que por victorias (entre los 2,000 primeros por victorias):


,rango_pr,rango_victorias,nombre,n_participaciones,n_adjudicaciones,tasa_exito,rivales_vencidos
41358,41359,767,"VLN SERVICIOS PUBLICITARIOS INTEGRALES S,L",21,21,1.000000,0
40846,40847,1829,BAKER BOTTS LLP,12,12,1.000000,0
39850,39851,1048,ANDINA DE RADIODIFUSION S.A.C.,17,17,1.000000,0
40201,40202,1409,RADIO LA EXITOSA S.A.C.,14,14,1.000000,0
38636,38637,235,GRUPORPP SOCIEDAD ANONIMA CERRADA,43,42,0.976744,0
38596,38597,252,GOOGLE LLC,40,40,1.000000,0
38516,38517,208,META PLATFORMS IRELAND LIMITED,45,45,1.000000,0
38005,38006,38,TIKTOK INC,110,110,1.000000,0


### Lectura
- PageRank **se parece a contar victorias** (correlación de rangos alta), pero **no es lo mismo**: premia ganar donde compiten muchos rivales activos y que a su vez ganan, y no premia tanto las victorias en procesos con un solo competidor.
- El **1% de arriba reúne una parte desproporcionada** del PageRank total: el voto se concentra en pocos proveedores.
- **Por qué sube o baja** (columna `rivales_vencidos`: cuántos proveedores distintos perdieron frente a él):
  - **Suben** los que, con pocas victorias, **vencieron a muchísimos rivales distintos**: ganaron procesos muy concurridos. Además, el voto de un rival que pierde poco pesa más para quien le ganó.
  - **Bajan** los que tienen **tasa de éxito cercana a 100%**: ganan procesos donde casi no hay rivales (por ejemplo, contratos de publicidad con una sola oferta), así que reciben pocos votos.
- **Esto no indica irregularidades.** Una empresa puede ser grande, estar bien posicionada o especializarse. Es una lista para **revisar con más contexto**, no un veredicto.

## 5. Sensibilidad al factor de amortiguamiento β
**Qué se mide:** se recalcula PageRank con varios β y se compara con el ranking de β = 0.85.
- **Spearman:** correlación entre los dos rankings (1 = idénticos).
- **Top-10 / Top-50 en común:** cuántos de los 10 (o 50) primeros siguen siendo los mismos.
- **Iteraciones:** cuántas hacen falta para converger (con β cercano a 1, la iteración es más lenta).
- **Masa del 1% de arriba:** qué parte del PageRank total reúne el 1% de proveedores con más rango.

In [8]:
resultados = {}      # beta -> DataFrame (proveedor_idx, pr)
filas, conv = [], []
for b in BETAS:
    t0 = time.time()
    if b == BETA:
        r_b, c_b = pr, cambios
    else:
        r_b, c_b = pagerank_spark(b)
    resultados[b] = r_b.set_index("proveedor_idx").pr
    conv += [{"beta": b, "iteracion": i + 1, "cambio": c} for i, c in enumerate(c_b)]
    print(f"β = {b:.2f}: {len(c_b)} iteraciones, {time.time()-t0:.0f} s")

base = resultados[BETA]
orden_base = base.sort_values(ascending=False).index
for b in BETAS:
    r_b = resultados[b].reindex(base.index)
    orden_b = r_b.sort_values(ascending=False).index
    filas.append({
        "beta": b,
        "iteraciones": len([c for c in conv if c["beta"] == b]),
        "convergio": "sí" if conv[[i for i, c in enumerate(conv) if c["beta"] == b][-1]]["cambio"] < TOL else "no",
        "spearman_vs_085": round(base.corr(r_b, method="spearman"), 4),
        "top10_en_comun": len(set(orden_base[:10]) & set(orden_b[:10])),
        "top50_en_comun": len(set(orden_base[:50]) & set(orden_b[:50])),
        "masa_top1pct": round(float(r_b.sort_values(ascending=False).head(int(0.01 * N)).sum()), 4),
        "pr_maximo_veces_uniforme": round(float(r_b.max() * N), 1),
    })
sensibilidad = pd.DataFrame(filas)
sensibilidad

β = 0.50: 24 iteraciones, 16 s


β = 0.70: 45 iteraciones, 31 s


β = 0.80: 70 iteraciones, 46 s
β = 0.85: 95 iteraciones, 0 s


β = 0.90: 145 iteraciones, 91 s


β = 0.95: 295 iteraciones, 175 s


,beta,iteraciones,convergio,spearman_vs_085,top10_en_comun,top50_en_comun,masa_top1pct,pr_maximo_veces_uniforme
0,0.50,24,sí,0.9956,8,36,0.0828,110.1
1,0.70,45,sí,0.9988,9,43,0.1225,195.4
2,0.80,70,sí,0.9998,9,46,0.1496,264.9
3,0.85,95,sí,1.0000,10,50,0.1672,312.8
4,0.90,145,sí,0.9997,10,46,0.1908,374.6
5,0.95,295,sí,0.9981,8,37,0.2299,451.8


### Lectura
- **El ranking es estable en la parte de arriba** para β entre 0.50 y 0.95: la correlación de rangos con β = 0.85 es mayor a 0.99 y entre 8 y 10 de los 10 primeros se mantienen.
- **Todos los β convergen** (columna `convergio`), pero cuanto mayor es β más iteraciones hacen falta: 24 con β = 0.50, 95 con β = 0.85 y unas 300 con β = 0.95. Con casi nada de teletransportación, el rango tarda en repartirse. Un valor extremo como 0.99 exigiría más de mil iteraciones y no se incluyó.
- **Cuanto mayor β, más se concentra el rango** en los de arriba (la teletransportación reparte menos) y **más iteraciones** se necesitan.
- **Cuanto menor β, más «plano»** es el reparto: se parece más a un reparto uniforme.
- Por eso se usa **β = 0.85**, el valor de clase: equilibra la señal de la red y la estabilidad del cálculo.

## 6. HITS sobre el grafo entidad → proveedor
**HITS** (*Hubs and Authorities*) da **dos puntajes** a cada nodo en un grafo con dirección:
- **Authority (autoridad):** nodo al que **apuntan buenos hubs**. Aquí: **proveedores** a los que adjudican entidades importantes.
- **Hub (concentrador):** nodo que **apunta a buenas authorities**. Aquí: **entidades** que adjudican a muchos proveedores relevantes.

**Grafo:** entidad → proveedor si la entidad adjudicó al proveedor al menos una vez en *train* (matriz del notebook 1, **sin pesos**: importa si hubo vínculo, no cuántas veces).

**Cálculo (iteración de potencia):** `autoridad = Aᵀ · hub` y `hub = A · autoridad`, normalizando en cada paso. Es el mismo tipo de iteración que PageRank.

**Dato útil:** la autoridad converge al **primer vector singular** de la matriz entidad×proveedor (la SVD de clase). Se verifica abajo.

In [9]:
arc_h = train.select(F.col("entidad_idx").alias("h"), F.col("proveedor_idx").alias("a")).distinct().cache()
ents  = arc_h.select("h").distinct().cache()
provs = arc_h.select("a").distinct().cache()
n_e, n_p = ents.count(), provs.count()
print(f"entidades (hubs): {n_e:,} | proveedores (authorities): {n_p:,} | enlaces: {arc_h.count():,}")

def normalizar(df, col):
    norma = df.agg(F.sqrt(F.sum(F.col(col) * F.col(col)))).first()[0]
    return df.withColumn(col, F.col(col) / F.lit(norma))

hub = ents.withColumn("s", F.lit(1.0)); hub = normalizar(hub, "s").localCheckpoint()
auth = provs.withColumn("s", F.lit(1.0)); auth = normalizar(auth, "s").localCheckpoint()
cambios_h = []
for it in range(1, MAX_ITER + 1):
    a_nuevo = arc_h.join(hub, "h").groupBy("a").agg(F.sum("s").alias("s"))
    a_nuevo = normalizar(a_nuevo, "s").localCheckpoint()
    h_nuevo = arc_h.join(a_nuevo, "a").groupBy("h").agg(F.sum("s").alias("s"))
    h_nuevo = normalizar(h_nuevo, "s").localCheckpoint()
    cambio = (auth.alias("x").join(a_nuevo.alias("y"), "a").agg(F.sum(F.abs(F.col("x.s") - F.col("y.s")))).first()[0])
    cambios_h.append(cambio)
    auth, hub = a_nuevo, h_nuevo
    if cambio < 1e-9:
        break
print(f"HITS: {len(cambios_h)} iteraciones, cambio final {cambios_h[-1]:.2e}")
auth_pd = auth.toPandas().rename(columns={"a": "proveedor_idx", "s": "autoridad"})
hub_pd  = hub.toPandas().rename(columns={"h": "entidad_idx", "s": "hub"})

entidades (hubs): 2,331 | proveedores (authorities): 9,833 | enlaces: 44,205


HITS: 80 iteraciones, cambio final 8.65e-10


In [10]:
# Verificación: la autoridad debe coincidir con el primer vector singular de la matriz entidad×proveedor
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import svds
tr = arc_h.toPandas()
ie = {e: i for i, e in enumerate(sorted(tr.h.unique()))}; ip = {p: i for i, p in enumerate(sorted(tr.a.unique()))}
A = coo_matrix((np.ones(len(tr)), (tr.h.map(ie), tr.a.map(ip))), shape=(len(ie), len(ip))).tocsr()
u, s, vt = svds(A, k=1)
v1 = np.abs(vt[0])
auth_vec = auth_pd.set_index("proveedor_idx").autoridad.reindex(sorted(ip, key=ip.get)).values
coseno = float(v1 @ auth_vec / (np.linalg.norm(v1) * np.linalg.norm(auth_vec)))
print(f"coseno entre autoridad (HITS) y primer vector singular (SVD): {coseno:.6f}")
assert coseno > 0.999

coseno entre autoridad (HITS) y primer vector singular (SVD): 1.000000


In [11]:
# Interpretación: proveedores authority y entidades hub
ed = spark.read.parquet(f"{RUTA_P2}/entidad_dim").select("entidad_idx", F.col("nombre").alias("entidad"), "entidad_tipo", "entidad_departamento").toPandas()
n_prov_por_ent = tr.groupby("h").a.nunique().rename("n_proveedores")
n_ent_por_prov = tr.groupby("a").h.nunique().rename("n_entidades_train")

aut = auth_pd.merge(feat, on="proveedor_idx", how="left").merge(n_ent_por_prov, left_on="proveedor_idx", right_index=True, how="left")
aut["rango_autoridad"] = aut.autoridad.rank(ascending=False, method="first").astype(int)
aut = aut.merge(t[["proveedor_idx", "rango_pr"]], on="proveedor_idx", how="left").sort_values("rango_autoridad")
hu = hub_pd.merge(ed, on="entidad_idx", how="left").merge(n_prov_por_ent, left_on="entidad_idx", right_index=True, how="left")
hu["rango_hub"] = hu.hub.rank(ascending=False, method="first").astype(int)
hu = hu.sort_values("rango_hub")

print("Correlación de rangos (Spearman):")
print("  autoridad vs número de entidades:", round(aut.autoridad.corr(aut.n_entidades_train, method="spearman"), 3))
comun = aut.dropna(subset=["rango_pr"])
print("  autoridad vs PageRank (proveedores en ambos grafos):", round(comun.autoridad.corr(-comun.rango_pr, method="spearman"), 3), f"({len(comun):,} proveedores)")
print("  hub vs número de proveedores:", round(hu.hub.corr(hu.n_proveedores, method="spearman"), 3))
print(f"el 1% de proveedores con más autoridad reúne {100*(aut.autoridad.head(int(0.01*len(aut)))**2).sum():.1f}% de la energía total")
display(aut.head(10)[["rango_autoridad", "nombre", "ruc", "n_entidades_train", "rango_pr"]])
hu.head(10)[["rango_hub", "entidad", "entidad_tipo", "entidad_departamento", "n_proveedores"]]

Correlación de rangos (Spearman):
  autoridad vs número de entidades: 0.525
  autoridad vs PageRank (proveedores en ambos grafos): 0.238 (9,356 proveedores)
  hub vs número de proveedores: 0.848
el 1% de proveedores con más autoridad reúne 46.8% de la energía total


,rango_autoridad,nombre,ruc,n_entidades_train,rango_pr
2452,1,DIAGNOSTICA PERUANA S.A.C.,20501887286,96,12.0
35,2,AMERICA MOVIL PERU S.A.C.,20467534026,151,4.0
4922,3,MAPFRE PERU COMPAÑIA DE SEGUROS Y REASEGUROS,20418896915,155,88.0
150,4,SIMED PERU S.A.C.,20553853355,53,87.0
2461,5,LINDE PERU S.R.L.,20338570041,50,1373.0
7318,6,LABORATORIOS AC FARMA S.A.,20347268683,48,18.0
7386,7,B.BRAUN MEDICAL PERU S.A.,20377339461,41,74.0
2473,8,NIPRO MEDICAL CORPORATION SUCURSAL DEL PERU,20504312403,35,103.0
89,9,SISTEMAS ANALITICOS SRL,20155695901,43,200.0
5038,10,DROGUERIA CADILLO S.A.C,20392764373,45,96.0


,rango_hub,entidad,entidad_tipo,entidad_departamento,n_proveedores
1830,1,SEGURO SOCIAL DE SALUD,salud,LIMA,701
75,2,CENTRO NACIONAL DE ABASTECIMIENTO DE RECURSOS ...,salud,LIMA,176
726,3,DIRECCIÓN DE SANIDAD DE LA PNP,otro,LIMA,174
1778,4,INSTITUTO NACIONAL DE ENFERMEDADES NEOPLASICAS,otro,LIMA,166
109,5,INSTITUTO NACIONAL DE SALUD DEL NIÑO - SAN BORJA,salud,LIMA,124
727,6,HOSPITAL DE EMERGENCIAS VILLA EL SALVADOR,salud,LIMA,100
1208,7,INSTITUTO NACIONAL DE SALUD DEL NIÑO,salud,LIMA,108
12,8,HOSPITAL NACIONAL DOS DE MAYO,salud,LIMA,113
641,9,FUERZA AEREA DEL PERU,defensa_policia,LIMA,216
1308,10,IAFAS DEL EJERCITO DEL PERU (FOSPEME),defensa_policia,LIMA,98


### Lectura
- **Autoridades:** proveedores con adjudicaciones de **entidades que a su vez adjudican a muchos proveedores relevantes**. Tiene relación moderada con el número de entidades con que trabaja (correlación de rangos cercana a 0.5): pesa **cuáles** son esas entidades, no solo cuántas.
- **Hubs:** entidades grandes, con muchos proveedores (correlación de rangos con el número de proveedores cercana a 0.85); en esta base son sobre todo hospitales, institutos de salud y entidades de defensa de Lima.
- **PageRank y HITS responden preguntas distintas:** PageRank mira **quién gana frente a quién** (competencia); HITS mira **quién compra a quién** (relación entidad–proveedor). Su correlación de rangos es baja (cerca de 0.24): un proveedor puede destacar en una y no en la otra.
- La autoridad coincide con la SVD de clase: es la **dirección dominante** de la matriz entidad×proveedor. Eso conecta la Parte IV con la representación de la Parte VI.
- Igual que antes: una autoridad alta **no indica irregularidades**; refleja volumen y diversidad de relaciones.

## 7. Archivos para la página web
**Datos personales:** igual que en el resto del proyecto, el RUC que no empieza con 20 se muestra enmascarado (`10283****79`) y el nombre de una persona natural (RUC 10, 15, 16 o 17) se reemplaza por «Persona natural». Los nombres de entidades públicas se muestran tal cual.

In [12]:
def enmascarar(ruc):
    r = str(ruc); return r if r.startswith("20") else r[:5] + "****" + r[-2:]
def nombre_visible(ruc, nombre):
    r = str(ruc); return f"Persona natural (RUC {enmascarar(r)})" if r[:2] in ("10", "15", "16", "17") else nombre
def publico(df):
    d = df.copy()
    d["nombre"] = [nombre_visible(r, n) for r, n in zip(d.ruc, d.nombre)]
    d["ruc"] = d.ruc.map(enmascarar)
    return d

# 1) Top 30 de PageRank (β = 0.85)
cols = ["rango_pr", "nombre", "ruc", "pr_veces_uniforme", "n_participaciones", "n_adjudicaciones", "tasa_exito", "n_entidades", "rango_victorias", "rivales_vencidos", "categoria_principal"]
top30 = publico(t.head(30)[cols]); top30["tasa_exito"] = top30.tasa_exito.round(3)
guardar_csv("pr_top.csv", top30)

# 2) Top 25 para cada β (el slider de la página elige β)
filas = []
meta = t.set_index("proveedor_idx")[["nombre", "ruc"]]
for b in BETAS:
    s = resultados[b].sort_values(ascending=False).head(25)
    for k, (idx, v) in enumerate(s.items(), 1):
        filas.append({"beta": b, "rango": k, "proveedor_idx": idx, "nombre": meta.loc[idx, "nombre"], "ruc": meta.loc[idx, "ruc"], "pr_veces_uniforme": round(v * N, 1)})
por_beta = publico(pd.DataFrame(filas)).drop(columns="proveedor_idx")
guardar_csv("pr_por_beta.csv", por_beta)

# 3) Sensibilidad y convergencia
guardar_csv("pr_sensibilidad.csv", sensibilidad)
conv_df = pd.DataFrame(conv); conv_df["cambio"] = conv_df.cambio.map(lambda x: float(f"{x:.3e}"))
guardar_csv("pr_convergencia.csv", conv_df)

# 4) Dispersión PageRank contra victorias (sin nombres): los 300 primeros y una muestra de 3,000 del resto
muestra = pd.concat([t.head(300), t.iloc[300:].sample(3000, random_state=42)])
guardar_csv("pr_dispersion.csv", muestra[["rango_pr", "pr_veces_uniforme", "n_participaciones", "n_adjudicaciones", "tasa_exito", "rivales_vencidos"]].round(4))

# 5) HITS
a_web = publico(aut.head(20)[["rango_autoridad", "nombre", "ruc", "n_entidades_train", "rango_pr", "n_adjudicaciones"]].rename(columns={"n_entidades_train": "n_entidades"}))
guardar_csv("hits_autoridades.csv", a_web)
h_web = hu.head(20)[["rango_hub", "entidad", "entidad_tipo", "entidad_departamento", "n_proveedores"]]
guardar_csv("hits_hubs.csv", h_web)
guardar_csv("hits_convergencia.csv", pd.DataFrame({"iteracion": range(1, len(cambios_h) + 1), "cambio": [float(f"{c:.3e}") for c in cambios_h]}))

# 6) Resumen
guardar_json("p4_resumen.json", {
    "grafo_pagerank": {
        "descripcion": "Arista A → B: A ofertó y no ganó, B ganó en el mismo proceso. Peso = procesos en que ocurrió.",
        "nodos": N, "aristas": n_arcos, "dead_ends": n_sin_salida, "sin_entrada": n_sin_entrada,
        "pct_dead_ends": round(100 * n_sin_salida / N, 1), "pct_sin_entrada": round(100 * n_sin_entrada / N, 1),
        "parametros": {"min_participaciones": MIN_PART, "tope_postores": TOPE, "beta": BETA},
    },
    "pagerank": {
        "iteraciones": len(cambios), "diferencia_vs_networkx": float(f"{dif_max:.2e}"),
        "masa_top1pct": round(float(masa_top), 4), "n_top1pct": int(0.01 * N),
        "spearman_con_victorias": round(t.pr.corr(t.n_adjudicaciones, method="spearman"), 3),
        "spearman_con_participaciones": round(t.pr.corr(t.n_participaciones, method="spearman"), 3),
        "spearman_con_tasa_exito": round(t.pr.corr(t.tasa_exito, method="spearman"), 3),
    },
    "hits": {
        "entidades": n_e, "proveedores": n_p, "enlaces": int(len(tr)), "iteraciones": len(cambios_h),
        "coseno_con_svd": round(coseno, 6),
        "spearman_autoridad_vs_n_entidades": round(aut.autoridad.corr(aut.n_entidades_train, method="spearman"), 3),
        "spearman_autoridad_vs_pagerank": round(comun.autoridad.corr(-comun.rango_pr, method="spearman"), 3),
        "spearman_hub_vs_n_proveedores": round(hu.hub.corr(hu.n_proveedores, method="spearman"), 3),
    },
})
spark.stop()

guardado: pr_top.csv 30 filas
guardado: pr_por_beta.csv 150 filas
guardado: pr_sensibilidad.csv 6 filas
guardado: pr_convergencia.csv 674 filas
guardado: pr_dispersion.csv 3300 filas
guardado: hits_autoridades.csv 20 filas
guardado: hits_hubs.csv 20 filas
guardado: hits_convergencia.csv 80 filas
guardado: p4_resumen.json
